In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset
import numpy as np
import torch
from torchvision import transforms as T

import os

def preprocess_normal_for_cosine(normal_rgb, fg_mask=None, bg_color=None, bg_eps=0.02):
    """
    Preprocess normal map for cosine similarity loss.

    The normal PNG has two invalid regions:
      (1) uncropping padding outside the pixel3dmm bbox (constant color)
      (2) around-head pixels inside the bbox that pixel3dmm wrote garbage into
          (excluded via the FG/rembg silhouette mask)

    Args:
        normal_rgb: [H, W, 3] or [3, H, W], values in [0, 1] (standard normal map: RGB = xyz)
        fg_mask: optional [1, H, W] or [H, W] foreground (rembg) mask in [0, 1].
            If provided, the validity mask is intersected with it, removing
            inside-bbox-but-not-head pixels (shoulders, neck cutoff, in-crop bg, etc).
        bg_color: optional [3] iterable in [0, 1] specifying the uncropping padding color.
            If None, it is auto-detected as the median of the four corner pixels.
        bg_eps: max L_inf distance (in [0,1] RGB space) from bg_color to count as padding.

    Returns:
        normal: [3, H, W] L2-normalized, in [-1, 1] range (zeroed where invalid)
        valid_mask: [1, H, W] float, 1 where valid (head region), 0 elsewhere
    """
    if isinstance(normal_rgb, np.ndarray):
        normal = torch.from_numpy(normal_rgb).float()
    else:
        normal = normal_rgb.float()

    if normal.ndim == 3 and normal.shape[0] != 3 and normal.shape[-1] == 3:
        normal = normal.permute(2, 0, 1)  # HWC -> CHW

    # (1) Padding mask: detect padding color from four corners (median is robust to
    # one corner accidentally containing face).
    if bg_color is None:
        corners = torch.stack([
            normal[:, 0, 0],
            normal[:, 0, -1],
            normal[:, -1, 0],
            normal[:, -1, -1],
        ], dim=0)  # [4, 3]
        bg_color = corners.median(dim=0).values  # [3]
    else:
        bg_color = torch.as_tensor(bg_color, dtype=normal.dtype)

    diff = (normal - bg_color.view(3, 1, 1)).abs().amax(dim=0, keepdim=True)  # [1,H,W]
    in_bbox_mask = (diff > bg_eps).float()  # 1 inside the pixel3dmm crop, 0 in padding

    # (2) FG mask: removes inside-bbox-but-not-head pixels
    if fg_mask is not None:
        if isinstance(fg_mask, np.ndarray):
            fg_mask = torch.from_numpy(fg_mask).float()
        fg_mask = fg_mask.float()
        if fg_mask.ndim == 2:
            fg_mask = fg_mask.unsqueeze(0)  # [1, H, W]
        fg_bin = (fg_mask > 0.5).float()
        valid_mask = in_bbox_mask * fg_bin
    else:
        valid_mask = in_bbox_mask

    # RGB [0, 1] -> xyz [-1, 1], then L2-normalize per pixel
    normal = normal * 2.0 - 1.0
    normal = normal / normal.norm(dim=0, keepdim=True).clamp(min=1e-8)
    normal = normal * valid_mask  # zero out invalid pixels

    return normal, valid_mask

def preprocess_image(input_im):
    '''
    :param input_im (PIL Image).
    :return input_im (H, W, 3) array in [0, 1].
    '''
    input_im = input_im.resize([256, 256], Image.Resampling.LANCZOS)
    input_im = np.asarray(input_im, dtype=np.float32) / 255.0
    # (H, W, 4) array in [0, 1].

    # old method: thresholding background, very important
    # input_im[input_im[:, :, -1] <= 0.9] = [1., 1., 1., 1.]

    # new method: apply correct method of compositing to avoid sudden transitions / thresholding
    # (smoothly transition foreground to white background based on alpha values)
    if input_im.shape[-1] == 4:
        alpha = input_im[:, :, 3:4]
        white_im = np.ones_like(input_im)
        input_im = alpha * input_im + (1.0 - alpha) * white_im

    input_im = input_im[:, :, 0:3]
    # (H, W, 3) array in [0, 1].

    return input_im

def _normal_to_rgb_vis(norm):
    """Map unit normal field to RGB in [0,1] for visualization. norm: [B,3,H,W] or [3,H,W]. Returns [3,H,W]."""
    if norm.dim() == 4:
        norm = norm[0]
    return torch.clamp((norm + 1) / 2, 0, 1)

class Portrait4dDataset(Dataset):
    def __init__(self, root_dir, transform=None, src_view_idx=9, skip_identity=True,
                 load_normals=True, load_depth=False):
        super().__init__()
        self.root_dir = root_dir
        self.transform = transform
        self.src_view_idx = src_view_idx
        self.skip_identity = skip_identity
        self.load_normals = load_normals
        self.load_depth = load_depth

        self.subjects = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])
        self.samples = []
        skipped = 0

        for sub in self.subjects:
            sub_path = os.path.join(root_dir, sub)
            poses_path = os.path.join(sub_path, 'poses.npy')
            if not os.path.exists(poses_path):
                skipped += 1
                continue

            views = sorted([
                f for f in os.listdir(sub_path)
                if os.path.isfile(os.path.join(sub_path, f))
                and os.path.splitext(f)[1].lower() in [".jpg", ".png"]
                and os.path.splitext(f)[0].isdigit()
            ])
            # views = sorted([f for f in os.listdir(sub_path)
            #                if f.endswith('.jpg') and not f.startswith('src')])

            for view_file in views:
                view_id = view_file.split('.')[0]
                view_index = int(view_id)
                if self.skip_identity and view_index == self.src_view_idx:
                    continue

                mask_file = f"{view_id}_mask.png"
                normal_file = os.path.join("normals_uncropped", f"{view_id}.png")
                normal_path = os.path.join(sub_path, normal_file)
                depth_file = os.path.join("depth", f"{view_id}.npy")
                depth_path = os.path.join(sub_path, depth_file)

                has_mask = os.path.exists(os.path.join(sub_path, mask_file))
                has_normal = os.path.exists(normal_path) if self.load_normals else True
                has_depth = os.path.exists(depth_path) if self.load_depth else True

                if has_mask and has_normal and has_depth:
                    self.samples.append({
                        'sub_path': sub_path,
                        'view_file': view_file,
                        'mask_file': mask_file,
                        'normal_file': normal_file,
                        'depth_file': depth_file,
                        'view_index': view_index,
                    })

        parts = []
        if self.load_normals:
            parts.append("normals")
        if self.load_depth:
            parts.append("depth")
        extra_str = f" (with {', '.join(parts)})" if parts else ""
        print(f"[Dataset] Loaded {len(self.samples)} samples{extra_str}, skipped {skipped} incomplete subjects")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        sub_path = sample['sub_path']

        src_path = os.path.join(sub_path, f"{self.src_view_idx:05d}.png")
        src = Image.open(src_path).convert("RGBA")

        img_path = os.path.join(sub_path, sample['view_file'])
        image = Image.open(img_path).convert("RGBA")

        mask_path = os.path.join(sub_path, sample['mask_file'])
        mask = Image.open(mask_path).convert("L")

        poses = np.load(os.path.join(sub_path, 'poses.npy'))
        if poses.ndim == 2 and poses.shape[1] == 16:
            poses = poses.reshape(-1, 4, 4)

        delta_pose = compute_relative_pose(
            poses[self.src_view_idx],
            poses[sample['view_index']]
        )
        assert abs(delta_pose[1]) < 1.0, f"Yaw {delta_pose[1]} seems too large - check units!"
        delta_pose = torch.from_numpy(delta_pose).float()

        src = preprocess_image(src) # Returns [H, W, 3] numpy array
        image = preprocess_image(image) # Returns [H, W, 3] numpy array

        src = torch.from_numpy(src).permute(2, 0, 1).float()
        image = torch.from_numpy(image).permute(2, 0, 1).float()

        if self.transform:
            src = self.transform(src)
            image = self.transform(image)

        mask_transform = T.Compose([T.Resize((256, 256)), T.ToTensor()])
        mask = mask_transform(mask)

        out = {
            "jpg": image,
            "hint": src,
            "mask": mask,
            "delta_pose": delta_pose,
            "subject_id": os.path.basename(sub_path),
            "txt": ""
        }

        if self.load_normals:
            normal_path = os.path.join(sub_path, sample['normal_file'])
            normal_img = Image.open(normal_path).convert("RGB")
            # NEAREST avoids bleeding padding color into face pixels along the boundary
            normal_img = normal_img.resize((256, 256), Image.NEAREST)
            normal_np = np.array(normal_img).astype(np.float32) / 255.0
            # Two-stage masking:
            #   (1) padding-color detection removes the uncropped boundary
            #   (2) FG (rembg) mask removes inside-bbox-but-not-head pixels
            fg_for_normal = mask[0].numpy()  # [H, W] in [0, 1]
            normal, normal_valid_mask = preprocess_normal_for_cosine(
                normal_np, fg_mask=fg_for_normal
            )
            out["normal"] = normal
            out["normal_mask"] = normal_valid_mask

        if self.load_depth:
            depth_path = os.path.join(sub_path, sample['depth_file'])
            depth_np = np.load(depth_path).astype(np.float32)
            if depth_np.ndim == 3:  # (H, W, 1) or (1, H, W) -> (H, W)
                depth_np = depth_np.squeeze()
            # Sanitize BEFORE interpolation: bilinear weights spread NaN to neighbors
            depth_np = np.nan_to_num(depth_np, nan=0.0, posinf=0.0, neginf=0.0)
            depth_t = torch.from_numpy(depth_np).float().unsqueeze(0)  # [1, H, W]
            depth_t = torch.nn.functional.interpolate(
                depth_t.unsqueeze(0), size=(256, 256),
                mode='bilinear', align_corners=False
            ).squeeze(0)  # [1, H, W]
            depth_t = torch.clamp(depth_t, min=0.0)  # ensure non-negative
            valid = (depth_t > 1e-6) & (depth_t < 1e6)
            depth_mask = valid.float()
            out["depth"] = depth_t
            out["depth_mask"] = depth_mask

        return out

from torchvision import transforms as T
from torch.utils.data import Dataset, DataLoader

transform = T.Compose([
    T.Resize((256, 256))
])

DATASET_ROOT = "/content/drive/MyDrive/datasets/TOSS_Dataset/portrait4d_rembg"
TRAIN_ROOT = os.path.join(DATASET_ROOT, "train")
TEST_ROOT = os.path.join(DATASET_ROOT, "test")
# dataset_name = "portrait4d_rembg_isnet"

train_dataset = Portrait4dDataset(root_dir=TRAIN_ROOT, transform=transform)
test_dataset  = Portrait4dDataset(root_dir=TEST_ROOT, transform=transform)
print(f"train identities ({len(train_dataset.subjects)}):", train_dataset.subjects)
print(f"test  identities ({len(test_dataset.subjects)}):", test_dataset.subjects)

dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)

data_iter = iter(dataloader)
batch = next(data_iter)
print("Batch keys:", batch.keys())


In [ ]:
import matplotlib.pyplot as plt

def visualize_normal_and_mask(sample, title=None, show_raw=True):
    """Visualize the two-stage normal mask:
       (1) in_bbox_mask: pixels NOT matching the uncropping padding color
       (2) fg_mask:      rembg silhouette
       final normal_mask = (1) AND (2)
    """
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))

    rgb = sample["jpg"].permute(1, 2, 0).numpy()
    axes[0, 0].imshow(np.clip(rgb, 0, 1))
    axes[0, 0].set_title("RGB (jpg)")
    axes[0, 0].axis("off")

    fg_mask = sample["mask"][0].numpy()
    axes[0, 1].imshow(fg_mask, cmap="gray", vmin=0, vmax=1)
    axes[0, 1].set_title(f"FG mask (rembg)\nmean={fg_mask.mean():.3f}")
    axes[0, 1].axis("off")

    raw_normal = None
    bg_color = None
    in_bbox_mask = None
    if show_raw and "view_index" in sample:
        sub_path = os.path.join(TRAIN_ROOT, sample["subject_id"])
        view_id = f"{int(sample['view_index']):05d}"
        raw_normal_path = os.path.join(sub_path, "normals_uncropped", f"{view_id}.png")
        # NEAREST so corner pixels stay constant for accurate bg detection
        raw_normal = np.array(
            Image.open(raw_normal_path).convert("RGB").resize((256, 256), Image.NEAREST)
        ).astype(np.float32) / 255.0
        axes[0, 2].imshow(raw_normal)
        axes[0, 2].set_title("Raw normal PNG\n(NEAREST 256x256)")

        # Recompute the in-bbox (NOT padding) mask for visualization
        h, w = raw_normal.shape[:2]
        corner_arr = np.stack([
            raw_normal[0, 0], raw_normal[0, w - 1],
            raw_normal[h - 1, 0], raw_normal[h - 1, w - 1],
        ], axis=0)  # [4, 3]
        bg_color = np.median(corner_arr, axis=0)  # [3]
        diff = np.abs(raw_normal - bg_color[None, None, :]).max(axis=-1)
        in_bbox_mask = (diff > 0.02).astype(np.float32)
    else:
        axes[0, 2].text(0.5, 0.5, "raw normal\nnot available", ha="center", va="center")
        axes[0, 2].set_title("Raw normal PNG")
    axes[0, 2].axis("off")

    if in_bbox_mask is not None:
        axes[0, 3].imshow(in_bbox_mask, cmap="gray", vmin=0, vmax=1)
        axes[0, 3].set_title(f"(1) in_bbox_mask\n(NOT padding color)\nmean={in_bbox_mask.mean():.3f}")
    else:
        axes[0, 3].text(0.5, 0.5, "n/a", ha="center", va="center")
        axes[0, 3].set_title("in_bbox_mask")
    axes[0, 3].axis("off")

    normal_vis = _normal_to_rgb_vis(sample["normal"]).permute(1, 2, 0).numpy()
    axes[1, 0].imshow(normal_vis)
    axes[1, 0].set_title("Preprocessed normal\n(unit xyz -> RGB)")
    axes[1, 0].axis("off")

    normal_mask = sample["normal_mask"][0].numpy()
    axes[1, 1].imshow(normal_mask, cmap="gray", vmin=0, vmax=1)
    axes[1, 1].set_title(f"final normal_mask\n(1) AND (2)\nvalid frac={normal_mask.mean():.3f}")
    axes[1, 1].axis("off")

    # RGB overlay: R = in_bbox (stage 1), G = FG mask (stage 2), B = final intersection
    if in_bbox_mask is not None:
        overlay = np.stack([in_bbox_mask, fg_mask, normal_mask], axis=-1)
        axes[1, 2].imshow(overlay)
        axes[1, 2].set_title("Overlay\nR=in_bbox  G=fg_mask  B=final")
    else:
        overlay = np.stack([normal_mask, fg_mask, np.zeros_like(normal_mask)], axis=-1)
        axes[1, 2].imshow(overlay)
        axes[1, 2].set_title("Overlay\nR=normal_mask  G=fg_mask")
    axes[1, 2].axis("off")

    # RGB applied to mask: shows what the loss actually sees
    masked_rgb = np.clip(rgb, 0, 1) * normal_mask[..., None]
    axes[1, 3].imshow(masked_rgb)
    axes[1, 3].set_title("RGB * final mask\n(what the loss supervises)")
    axes[1, 3].axis("off")

    if title:
        fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

    print(f"subject={sample['subject_id']}")
    if raw_normal is not None:
        h, w = raw_normal.shape[:2]
        corners = {
            "TL": raw_normal[0, 0], "TR": raw_normal[0, w - 1],
            "BL": raw_normal[h - 1, 0], "BR": raw_normal[h - 1, w - 1],
        }
        for name, px in corners.items():
            print(f"  raw corner {name}: rgb={px.round(3)}, mag={np.linalg.norm(px):.4f}")
        print(f"  detected padding color: rgb={bg_color.round(3)}")
        print(f"  in_bbox_mask valid fraction:  {in_bbox_mask.mean():.4f}  (after stage 1)")
    print(f"  fg_mask valid fraction:       {fg_mask.mean():.4f}")
    print(f"  final normal_mask valid frac: {normal_mask.mean():.4f}  (after stage 1 AND 2)")


num_vis = min(3, len(train_dataset))
for i in range(num_vis):
    sample = dict(train_dataset[i])
    sample["view_index"] = train_dataset.samples[i]["view_index"]
    visualize_normal_and_mask(
        sample,
        title=f"Sample {i} | subject={sample['subject_id']} | view={sample['view_index']}",
    )

if "normal" in batch and "normal_mask" in batch:
    print("\n--- From current dataloader batch (index 0) ---")
    batch_sample = {k: (v[0] if torch.is_tensor(v) else v[0]) for k, v in batch.items()}
    visualize_normal_and_mask(batch_sample, title="Dataloader batch[0]", show_raw=False)

In [ ]:
import numpy as np

def rotation_matrix_to_euler(R):
    """
    Extract pitch (x-rotation) and yaw (y-rotation) from a 3x3 rotation matrix.
    Returns angles in radians.

    Assumes rotation order: R = Ry(yaw) @ Rx(pitch) @ Rz(roll)
    """
    # Clamp to avoid numerical issues with asin
    sy = np.clip(R[0, 2], -1.0, 1.0)
    yaw = np.arcsin(sy)

    # Check for gimbal lock
    if np.abs(sy) < 0.99999:
        pitch = np.arctan2(-R[1, 2], R[2, 2])
    else:
        pitch = np.arctan2(R[2, 1], R[1, 1])

    return pitch, yaw


def pose_matrix_to_toss_format(pose_4x4):
    """
    Convert a 4x4 pose matrix to TOSS format: [pitch, yaw, distance]

    Args:
        pose_4x4: 4x4 transformation matrix (camera-to-world or world-to-camera)

    Returns:
        [pitch, yaw, distance] as expected by TOSS pose_enc="vae" or "freq"
    """
    R = pose_4x4[:3, :3]  # 3x3 rotation
    t = pose_4x4[:3, 3]   # translation vector

    pitch, yaw = rotation_matrix_to_euler(R)

    # Distance: typically the Z component or the norm of translation
    # Adjust based on your coordinate system
    distance = np.linalg.norm(t)  # or t[2] if Z is the depth axis

    return np.array([pitch, yaw, distance], dtype=np.float32)


def compute_relative_pose(src_pose_4x4, tgt_pose_4x4):
    """
    Compute relative pose from source to target view.
    Returns [delta_pitch, delta_yaw, delta_distance] in radians.

    Pitch uses R_tgt @ R_src^T to avoid ~±pi jumps on multi-view rigs.
    Yaw uses tgt_yaw - src_yaw (Portrait4d-style subtraction).
    """
    R_src = src_pose_4x4[:3, :3]
    R_tgt = tgt_pose_4x4[:3, :3]
    R_rel = R_tgt @ R_src.T

    delta_pitch, _ = rotation_matrix_to_euler(R_rel)
    _, src_yaw = rotation_matrix_to_euler(R_src)
    _, tgt_yaw = rotation_matrix_to_euler(R_tgt)
    delta_yaw = tgt_yaw - src_yaw

    src_dist = np.linalg.norm(src_pose_4x4[:3, 3])
    tgt_dist = np.linalg.norm(tgt_pose_4x4[:3, 3])
    delta_distance = tgt_dist - src_dist

    return np.array([delta_pitch, delta_yaw, delta_distance], dtype=np.float32)

# Debug: Check poses.npy format and values
import numpy as np

sub_path = "/content/drive/MyDrive/datasets/TOSS_Dataset/portrait4d_rembg_isnet/202483"
# sub_path = "datasets/portrait4d/202483"
poses = np.load(f"{sub_path}/poses.npy")

print(f"Poses shape: {poses.shape}")
print(f"Poses dtype: {poses.dtype}")

# Reshape if needed
if poses.ndim == 2 and poses.shape[1] == 16:
    poses = poses.reshape(-1, 4, 4)
    print(f"Reshaped to: {poses.shape}")

# Check a single pose matrix
print(f"\nPose[0] (view 00000):\n{poses[0]}")
print(f"\nPose[9] (view 00009 - center):\n{poses[9]}")

# Check if it looks like a valid rotation matrix
R = poses[0][:3, :3]
print(f"\nRotation matrix R (from pose[0]):\n{R}")
print(f"det(R) = {np.linalg.det(R):.6f} (should be ~1.0 for valid rotation)")
print(f"R @ R.T =\n{R @ R.T} (should be ~identity)")

# Extract angles using your function
pitch, yaw = rotation_matrix_to_euler(R)
print(f"\nExtracted angles from pose[0]:")
print(f"  pitch = {pitch:.4f} rad = {np.degrees(pitch):.2f}°")
print(f"  yaw = {yaw:.4f} rad = {np.degrees(yaw):.2f}°")

# Check all poses' yaw values
print(f"\nAll yaw values (degrees):")
for i in range(min(20, len(poses))):
    p, y = rotation_matrix_to_euler(poses[i][:3, :3])
    print(f"  View {i:02d}: pitch={np.degrees(p):+7.2f}°, yaw={np.degrees(y):+7.2f}°")

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint

checkpoint_cb = ModelCheckpoint(
    dirpath="/content/drive/MyDrive/checkpoints/toss_lora/",
    filename="toss-{step}",
    save_last=True,
    every_n_train_steps=500
)

trainer = pl.Trainer(
    max_steps=3000, # 1 step = 1 batch(4)
    accelerator="gpu",
    devices=1,
    callbacks=[checkpoint_cb],
    log_every_n_steps=10,
    gradient_clip_val=1.0,
    enable_checkpointing=True,
)

# Additional Config
model.learning_rate = 3e-4
model.sd_locked = True
model.first_stage_key = "jpg"
model.control_key = "hint"
model.cond_stage_key = "txt"

import torch.utils.checkpoint as cp
cp.checkpoint = lambda func, *args, **kwargs: func(*args)

trainer.fit(model, dataloader)

In [ ]:
"""
TOSS inference API for notebooks — same sampling pipeline as app.py (no Gradio, no CLI).

Example:
    from inference import TossInference
    runner = TossInference(resume_path="ckpt/toss.ckpt")
    out = runner.generate("input.png", prompt="a red shoe", dy=-90)
    out = runner.generate(pil_image, prompt="", dx=0, dy=0, dz=0)
"""
from __future__ import annotations

from pathlib import Path
from types import SimpleNamespace
from typing import Union

import numpy as np
import torch
from einops import rearrange
from omegaconf import OmegaConf
from PIL import Image
from pytorch_lightning import seed_everything
from torchvision import transforms

from ldm.models.diffusion.ddim import DDIMSampler

from app import get_T_from_relative, preprocess_image, sample_model

def load_model(device, _hparams, sd_locked, only_mid_control, cfgs):
    import os
    os.environ["WANDB_MODE"] = "disabled"
    model = instantiate_from_config(cfgs.model)

    # Load the state dict
    state_dict = torch.load("/content/drive/MyDrive/checkpoints/toss.ckpt", map_location="cpu", weights_only=False)["state_dict"]

    # --- FIX START ---
    # Remove the problematic key if it exists
    key_to_remove = "cond_stage_model.transformer.text_model.embeddings.position_ids"
    if key_to_remove in state_dict:
        print(f"Removing {key_to_remove} from state_dict to match model structure.")
        del state_dict[key_to_remove]
    # --- FIX END ---

    model.load_state_dict(state_dict, strict=False) # Adding strict=False is a safe backup

    trained_ckpt = torch.load("/content/drive/MyDrive/checkpoints/toss_lora/toss-step=3000-v4.ckpt", map_location="cpu", weights_only=False)

    lora_keys = [k for k in trained_ckpt["state_dict"].keys() if "lora" in k.lower()]
    print(f"Found {len(lora_keys)} LoRA keys in checkpoint:")
    for k in lora_keys:
        print(f"  {k}: {trained_ckpt['state_dict'][k].abs().mean().item():.6f}")

    # This will inject your LoRA weights and your trained PoseNet
    m, u = model.load_state_dict(trained_ckpt["state_dict"], strict=False)

    print()

    for n, p in model.named_parameters():
      if "lora" in n.lower() or ".out." in n.lower():
          print(n, p.abs().mean().item())

    print("Missing:", m)
    print("Unexpected keys (should be empty):", u)

    # reweight noise scheduler
    if _hparams.register_scheduler:
        model.register_schedule(given_betas=None, beta_schedule="linear", timesteps=1000, linear_start=0.00085, linear_end=0.016)

    model.learning_rate = _hparams.lr
    model.sd_locked = sd_locked
    model.only_mid_control = only_mid_control
    model = model.to(device)
    model.eval()
    return model

ImageInput = Union[str, Path, Image.Image, np.ndarray]


def _to_pil_rgba(image: ImageInput) -> Image.Image:
    if isinstance(image, (str, Path)):
        return Image.open(image).convert("RGBA")
    if isinstance(image, Image.Image):
        return image.convert("RGBA")
    if isinstance(image, np.ndarray):
        arr = image
        if arr.dtype != np.uint8:
            arr = np.clip(arr, 0.0, 1.0)
            if arr.max() <= 1.0:
                arr = (arr * 255.0).astype(np.uint8)
            else:
                arr = arr.astype(np.uint8)
        if arr.ndim == 2:
            raise ValueError("grayscale numpy arrays are not supported; use RGB/RGBA")
        if arr.shape[-1] == 4:
            return Image.fromarray(arr, mode="RGBA")
        if arr.shape[-1] == 3:
            return Image.fromarray(arr, mode="RGB").convert("RGBA")
        raise ValueError(f"expected HxWx3 or HxWx4 array, got shape {arr.shape}")
    raise TypeError(f"unsupported image type: {type(image)}")

class TossInference:
    """Load TOSS once, then call ``generate`` with varying inputs (notebook-friendly)."""

    def __init__(
        self,
        model_cfg: str | Path = "models/toss_vae.yaml",
        resume_path: str | Path = "/content/drive/MyDrive/checkpoints/toss.ckpt",
        *,
        device: torch.device | str | None = None,
        gpu: int = 0,
        register_scheduler: bool = False,
        lr: float = 1e-4,
        seed: int = 40,
        sd_locked: bool = True,
        only_mid_control: bool = False,
        use_ema_scope: bool = True,
        pose_enc: str = "freq",
        h: int = 256,
        w: int = 256,
    ):
        """
        Args:
            model_cfg: YAML defining the model (e.g. ``models/toss_vae.yaml``).
            resume_path: Checkpoint with ``state_dict`` (e.g. ``ckpt/toss.ckpt``).
            device: Explicit device; if None, uses ``cuda:{gpu}`` when available.
            gpu: CUDA index when ``device`` is None.
            register_scheduler: Passed through to ``load_model`` (same as training flag).
            lr: Required by ``load_model``; not used during inference.
            seed: Fixed at init; call ``set_seed`` to change between runs.
            use_ema_scope / pose_enc / h / w: Defaults for ``generate`` (overridable per call).
        """
        seed_everything(seed, workers=True)
        if device is None:
            self.device = torch.device(
                f"cuda:{gpu}" if torch.cuda.is_available() else "cpu"
            )
        else:
            self.device = torch.device(device)

        hparams = SimpleNamespace(
            resume_path=str(resume_path),
            register_scheduler=register_scheduler,
            lr=lr,
        )
        cfgs = OmegaConf.load(str(model_cfg))
        self.model = load_model(
            self.device, hparams, sd_locked, only_mid_control, cfgs
        )

        trainable = [n for n, p in self.model.named_parameters() if p.requires_grad]
        print("Trainable parameters:", len(trainable))
        for n in trainable:
            print(n)

        self.sampler = DDIMSampler(self.model)

        self._default_use_ema_scope = use_ema_scope
        self._default_pose_enc = pose_enc
        self._default_h = h
        self._default_w = w

    def set_seed(self, seed: int) -> None:
        """Call between ``generate`` runs for reproducible DDIM noise."""
        seed_everything(seed, workers=True)

    @torch.no_grad()
    def generate(
        self,
        image: ImageInput,
        prompt: str = "",
        dx: float = 0.0,
        dy: float = 0.0,
        dz: float = 0.0,
        *,
        pose_enc: str | None = None,
        h: int | None = None,
        w: int | None = None,
        precision: str = "fp32",
        n_samples: int = 1,
        use_ema_scope: bool | None = None,
        ddim_steps: int = 100,
        ddim_eta: float = 1.0,
        prompt_scale: float = 5.0,
        img_scale: float = 3.0,
        img_ucg: float = 0.05,
    ) -> Image.Image:
        """
        Novel view for one image (same logic as ``app.generate_loop_views`` / ``sample_model``).

        Args:
            image: Path, ``PIL.Image``, or ``HxWx3`` / ``HxWx4`` numpy array (float or uint8).
            prompt: Text conditioning (empty string allowed).
            dx, dy, dz: Relative pose in degrees / distance (see ``app.get_T_from_relative``).
        """
        h = self._default_h if h is None else h
        w = self._default_w if w is None else w
        pose_enc = self._default_pose_enc if pose_enc is None else pose_enc
        if use_ema_scope is None:
            use_ema_scope = self._default_use_ema_scope

        cond_im_pil = _to_pil_rgba(image)
        cond_im = preprocess_image(cond_im_pil)
        cond_im = transforms.ToTensor()(cond_im).unsqueeze(0).to(self.device)
        cond_im = transforms.functional.resize(cond_im, [h, w])

        T = get_T_from_relative(dx, dy, dz, pose_enc)
        x_samples = sample_model(
            cond_im,
            self.model,
            self.sampler,
            precision=precision,
            h=h,
            w=w,
            ddim_steps=ddim_steps,
            n_samples=n_samples,
            prompt_scale=prompt_scale,
            img_scale=img_scale,
            ddim_eta=ddim_eta,
            T=T,
            use_ema_scope=use_ema_scope,
            prompt=prompt,
            img_ucg=img_ucg,
        )
        assert x_samples.shape[0] == 1
        out = x_samples[0].cpu().numpy()
        out = 255.0 * rearrange(out, "c h w -> h w c")
        return Image.fromarray(out.astype(np.uint8))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math
import torch
from PIL import Image

sub_path = "/content/drive/MyDrive/datasets/TOSS_Dataset/portrait4d_rembg_isnet_v2/202525"
poses = np.load(f"{sub_path}/poses.npy").reshape(-1, 4, 4)

src_view_idx = 9
test_views = [8, 10, 0, 19]  # 가까운 2개, 먼 2개

src_img = Image.open(f"{sub_path}/{src_view_idx:05d}.png").convert("RGBA")

fig, axes = plt.subplots(len(test_views), 3, figsize=(12, len(test_views) * 3))
fig.suptitle("Source vs Generated vs GT", fontsize=14)

for row, view_idx in enumerate(test_views):
    # delta yaw 계산
    delta = compute_relative_pose(poses[src_view_idx], poses[view_idx])
    delta_yaw_deg = np.degrees(delta[1])

    # GT 이미지
    gt_img = Image.open(f"{sub_path}/{view_idx:05d}.png").convert("RGBA")
    gt_np = preprocess_image(gt_img)

    # Generate
    with torch.no_grad():
        gen_img = toss.generate(
            image=src_img,
            prompt="",
            dy=delta_yaw_deg,
        )

    # Source
    src_np = preprocess_image(src_img)
    axes[row, 0].imshow(src_np[:, :, :3])
    axes[row, 0].set_title(f"Source (view {src_view_idx})")
    axes[row, 0].axis("off")

    # Generated
    axes[row, 1].imshow(gen_img)
    axes[row, 1].set_title(f"Generated | Δyaw={delta_yaw_deg:.1f}°")
    axes[row, 1].axis("off")

    # GT
    axes[row, 2].imshow(gt_np[:, :, :3])
    axes[row, 2].set_title(f"GT (view {view_idx})")
    axes[row, 2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import torch
from PIL import Image

def compute_psnr(pred, target, data_range=1.0, eps=1e-10):
    """pred, target: float arrays in [0, 1], shape (H, W, 3)."""
    pred = np.asarray(pred, dtype=np.float32)
    target = np.asarray(target, dtype=np.float32)
    mse = float(np.mean((pred - target) ** 2))
    if mse <= eps:
        return float("inf")
    return float(20.0 * np.log10(data_range / np.sqrt(mse)))

src_view_idx = 9
test_views = [8, 10, 0, 19]

per_view_psnr = {v: [] for v in test_views}
per_subject_psnr = {s: [] for s in test_subject_filter}
all_psnr = []

for sub in test_subject_filter:
    sub_path = f"{DATA_ROOT}/{sub}"
    poses = np.load(f"{sub_path}/poses.npy").reshape(-1, 4, 4)
    src_img = Image.open(f"{sub_path}/{src_view_idx:05d}.png").convert("RGBA")

    for view_idx in test_views:
        delta = compute_relative_pose(poses[src_view_idx], poses[view_idx])
        delta_yaw_deg = float(np.degrees(delta[1]))

        with torch.no_grad():
            gen_pil = toss.generate(image=src_img, prompt="", dy=delta_yaw_deg)

        gt_pil = Image.open(f"{sub_path}/{view_idx:05d}.png").convert("RGBA")
        gt_np  = preprocess_image(gt_pil)
        gen_np = np.asarray(gen_pil, dtype=np.float32) / 255.0

        psnr = compute_psnr(gen_np, gt_np, data_range=1.0)
        per_view_psnr[view_idx].append(psnr)
        per_subject_psnr[sub].append(psnr)
        all_psnr.append(psnr)
        print(f"  subject {sub}, view {view_idx:02d}, dyaw={delta_yaw_deg:+.1f}°: PSNR = {psnr:.3f} dB")

print("\n=== Per-view mean PSNR ===")
for v in test_views:
    print(f"  view {v:02d}: {np.mean(per_view_psnr[v]):.3f} dB (n={len(per_view_psnr[v])})")

print("\n=== Per-subject mean PSNR ===")
for s in test_subject_filter:
    if per_subject_psnr[s]:
        print(f"  {s}: {np.mean(per_subject_psnr[s]):.3f} dB")

print(f"\n=== Overall mean PSNR: {np.mean(all_psnr):.3f} dB ===")

In [ ]:
# === Smoke test: DISTS + Sobel-DISTS perceptual loss building blocks ===
# Verifies the new pieces added to cldm/toss_lora.py work end-to-end
# (shape / range / grad flow) in this kernel, without instantiating the
# full TossLoraModule.
import torch

def _grayscale_sobel_3ch(rgb_01):
    from kornia.color import rgb_to_grayscale
    from kornia.filters import sobel
    gray = rgb_to_grayscale(rgb_01)
    edge = sobel(gray)
    return edge.expand(-1, 3, -1, -1).clamp(0.0, 1.0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# 1) Sobel helper: shape / range / grad
B, H, W = 2, 64, 64
rgb = torch.rand(B, 3, H, W, device=device, requires_grad=True)
edge = _grayscale_sobel_3ch(rgb)
assert edge.shape == (B, 3, H, W)
assert 0.0 <= edge.min().item() and edge.max().item() <= 1.0
edge.sum().backward()
assert rgb.grad is not None
print(f"[sobel] shape={tuple(edge.shape)} range=[{edge.min().item():.4f},{edge.max().item():.4f}] grad=ok")

# 2) DISTS forward+backward mirroring training_step usage
from DISTS_pytorch import DISTS
dists = DISTS().to(device)
for p in dists.parameters():
    p.requires_grad_(False)
dists.eval()

B, H, W = 2, 256, 256
pred = torch.rand(B, 3, H, W, device=device, requires_grad=True)
gt   = torch.rand(B, 3, H, W, device=device).detach()

d_img = dists(pred, gt, require_grad=True, batch_average=True)
pred_sobel = _grayscale_sobel_3ch(pred)
gt_sobel   = _grayscale_sobel_3ch(gt).detach()
d_sobel = dists(pred_sobel, gt_sobel, require_grad=True, batch_average=True)

dists_loss = d_img + d_sobel
print(f"[dists] d_img={d_img.item():.4f} d_sobel={d_sobel.item():.4f} dists_loss={dists_loss.item():.4f}")

dists_loss.backward()
assert pred.grad is not None and torch.isfinite(pred.grad).all()
print(f"[dists] pred.grad abs_mean={pred.grad.abs().mean().item():.3e}")
print(f"[dists] DISTS trainable params: {sum(1 for p in dists.parameters() if p.requires_grad)} (expect 0)")
print("SMOKE TEST OK")